<a href="https://colab.research.google.com/github/ahmad-goudah/alfaisalx-medmnist-challenge/blob/main/notebooks/AlfaisalX_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages

!pip -q install medmnist faiss-cpu scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.9 MB/s eta 0:00:00


In [2]:
# Imports + folders + seed

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

# Make sure folders exist in the repo structure
os.makedirs("models", exist_ok=True)
os.makedirs("reports/task1", exist_ok=True)
os.makedirs("reports/task3", exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [7]:
#Load PneumoniaMNIST dataset

import medmnist
from medmnist import PneumoniaMNIST
from torchvision import transforms

# Basic transforms (simple & safe)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    # light augmentation (optional but good)
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(28, scale=(0.9, 1.0)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = PneumoniaMNIST(split="train", transform=train_transform, download=True)
val_dataset   = PneumoniaMNIST(split="val", transform=test_transform, download=True)
test_dataset  = PneumoniaMNIST(split="test", transform=test_transform, download=True)

print("Train:", len(train_dataset),
      "Val:", len(val_dataset),
      "Test:", len(test_dataset))

#print("Num classes:", train_dataset.info["n_classes"])
print("Info dictionary:", train_dataset.info)
print("Task:", train_dataset.info["task"])
print("Image size:", train_dataset.info["n_channels"], "x", train_dataset.info["n_channels"], "(channel count shown separately)")





print("Dataset info keys:", train_dataset.info.keys())
# Try safe access
print("Task type:", train_dataset.info.get("task", "Not available"))
print("Number of classes:", train_dataset.info.get("label", "Binary classification"))
print("Number of channels:", train_dataset.info.get("n_channels", "Unknown"))

Train: 4708 Val: 524 Test: 624
Info dictionary: {'python_class': 'PneumoniaMNIST', 'description': 'The PneumoniaMNIST is based on a prior dataset of 5,856 pediatric chest X-Ray images. The task is binary-class classification of pneumonia against normal. We split the source training set with a ratio of 9:1 into training and validation set and use its source validation set as the test set. The source images are gray-scale, and their sizes are (384−2,916)×(127−2,713). We center-crop the images and resize them into 1×28×28.', 'url': 'https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1', 'MD5': '28209eda62fecd6e6a2d98b1501bb15f', 'url_64': 'https://zenodo.org/records/10519652/files/pneumoniamnist_64.npz?download=1', 'MD5_64': '8f4eceb4ccffa70c672198ea285246c6', 'url_128': 'https://zenodo.org/records/10519652/files/pneumoniamnist_128.npz?download=1', 'MD5_128': '05b46931834c231683c68f40c47b2971', 'url_224': 'https://zenodo.org/records/10519652/files/pneumoniamnist_224.npz

In [8]:
# Create Data Loaders

batch_size = 128  # good for CPU/GPU
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# Check one batch
images, labels = next(iter(train_loader))
print("Batch images:", images.shape)   # [B, 1, 28, 28]
print("Batch labels:", labels.shape)   # [B, 1]
print("Unique labels in batch:", labels.unique())